# CP3 Continual Learning

Notebook ini mendemonstrasikan implementasi CP3 untuk project Chicago Risk Score API: simulasi data baru berbasis waktu, deteksi drift, retraining dengan checkpoint, quality gate, versioning model, dan penyimpanan ringkasan hasil.

Notebook ini memakai data processed yang sudah tersedia di repository, jadi tidak bergantung pada raw notebook pipeline.

In [ ]:
cp3_config = {
    'batch_count': CP3_BATCH_COUNT,
    'drift_threshold': CP3_DRIFT_THRESHOLD,
    'quality_gate': CP3_QUALITY_GATE,
    'random_state': CP3_RANDOM_STATE,
    'max_rows_demo': 150000,
    'persist_outputs': False,
}
cp3_config

## 2. Definisikan Konfigurasi dan Parameter CP3

Parameter berikut dipakai sepanjang notebook untuk menyamakan perilaku simulasi dengan implementasi produksi di module CP3.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd

from app.config import BASE_DIR, CP3_BATCH_COUNT, CP3_DRIFT_THRESHOLD, CP3_QUALITY_GATE, CP3_RANDOM_STATE
from app.continual_learning import (
    append_model_version,
    build_version_entry,
    detect_drift,
    load_event_history,
    read_model_versions,
    run_cp3_pipeline,
    split_time_batches,
)

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 140)

PROJECT_ROOT = Path(BASE_DIR)
DATA_PATH = PROJECT_ROOT / 'data' / 'processed' / 'features_labels_events.csv'
OUTPUT_DIR = PROJECT_ROOT / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Project root: {PROJECT_ROOT}')
print(f'Data path   : {DATA_PATH}')

## 1. Persiapan Notebook dan Import Library

Bagian ini menyiapkan environment notebook, mengatur path kerja, dan mengimpor library yang diperlukan untuk simulasi CP3.

In [ ]:
pipeline_summary = run_cp3_pipeline(
    max_rows=cp3_config['max_rows_demo'],
    batch_count=cp3_config['batch_count'],
    drift_threshold=cp3_config['drift_threshold'],
    quality_gate=cp3_config['quality_gate'],
    persist=cp3_config['persist_outputs'],
)

print(json.dumps(pipeline_summary, indent=2, default=str)[:4000])

In [ ]:
simple_drift = detect_drift(10.0, 12.0, threshold=cp3_config['drift_threshold'])
edge_drift = detect_drift(10.0, 11.0, threshold=cp3_config['drift_threshold'])

assert simple_drift is True
assert edge_drift is False

print({'simple_drift': simple_drift, 'edge_drift': edge_drift})

In [ ]:
summary_path = OUTPUT_DIR / 'cp3_summary.json'
with summary_path.open('w', encoding='utf-8') as handle:
    json.dump(pipeline_summary, handle, indent=2, ensure_ascii=False, default=str)

print(f'Saved summary to: {summary_path}')
print('CP3 summary:')
print('1. Data processed dibagi menjadi batch temporal.')
print('2. Drift dideteksi dari kenaikan MAE.')
print('3. Candidate model hanya dipromosikan jika lolos quality gate.')
print('4. Versioning model disimpan ke model_versions.json.')

## 10. Simpan Output, Refactor, dan Ringkasan Implementasi

Hasil akhir disimpan sebagai JSON summary agar bisa dipakai ulang untuk laporan atau monitoring ringan. Bagian ini juga merangkum langkah utama CP3 yang sudah dibangun.

In [ ]:
batch_summary = pd.DataFrame(pipeline_summary['batches'])
drift_summary = pd.DataFrame(pipeline_summary['drift_events']) if pipeline_summary['drift_events'] else pd.DataFrame()
version_summary = pd.DataFrame(read_model_versions().get('model_versions', []))

print('Batch summary:')
display(batch_summary.head())

if not drift_summary.empty:
    print('Drift summary:')
    display(drift_summary)

print('Version summary:')
display(version_summary[['version', 'accepted', 'artifact_path', 'created_at']] if not version_summary.empty else version_summary)

## 9. Evaluasi Hasil dan Visualisasi

Hasil pipeline diringkas ke tabel agar mudah dibaca. Notebook ini tidak bergantung pada library visualisasi tambahan; ringkasan tabel sudah cukup untuk memeriksa kualitas checkpoint dan drift.

## 8. Uji dengan Kasus Sederhana dan Kasus Edge

Selain run utama, notebook ini mengeksekusi beberapa check ringan untuk memastikan drift logic dan quality gate bekerja pada nilai yang masuk akal.

In [ ]:
assert len(history) > 0, 'Processed history must not be empty.'
assert cp3_config['batch_count'] >= 2, 'Batch count must be at least 2.'
assert 0 < cp3_config['drift_threshold'] < 1, 'Drift threshold must be a ratio.'
assert 0 < cp3_config['quality_gate'] < 1, 'Quality gate must be a ratio.'
assert len(batches) >= 2, 'Temporal split must create at least two batches.'

print('All CP3 validation checks passed.')

## 7. Tambahkan Validasi Input dan Penanganan Error

Implementasi CP3 harus aman saat data tidak lengkap, batch terlalu sedikit, atau threshold salah. Cell berikut menunjukkan validasi dasar yang dipakai pipeline.

In [ ]:
initial_versions = read_model_versions()
final_versions = pipeline_summary['promoted_versions']
rejected_versions = pipeline_summary['rejected_candidates']

print(f'Versi awal   : {len(initial_versions.get("model_versions", []))}')
print(f'Dipromosikan : {len(final_versions)}')
print(f'Ditolak      : {len(rejected_versions)}')

## 6. Integrasikan Alur Proses Utama

Cell ini merangkai alur end-to-end dari input, pemecahan batch, simulasi drift, retraining, sampai pembentukan summary akhir.

## 5. Implementasi Fungsi Inti CP3

Fungsi inti CP3 sudah disiapkan di module [app/continual_learning.py](app/continual_learning.py). Notebook ini memanggil pipeline tersebut agar alur retraining, drift detection, dan checkpointing bisa dilihat end-to-end.

In [ ]:
batches = split_time_batches(history, batch_count=cp3_config['batch_count'])
model_versions_snapshot = read_model_versions()

batch_windows = [
    {
        'batch_index': index,
        'rows': len(batch),
        'start_date': batch['Date'].min(),
        'end_date': batch['Date'].max(),
    }
    for index, batch in enumerate(batches)
]

pd.DataFrame(batch_windows)

## 4. Bangun Struktur Data dan Helper Functions

Bagian ini memperlihatkan helper utama yang dipakai saat simulasi CP3, termasuk pemecahan batch waktu, evaluasi model, dan helper untuk menulis ringkasan versi model.

In [ ]:
history = load_event_history(max_rows=cp3_config['max_rows_demo'])
mock_input = {
    'datetime': '2026-07-30T14:00:00',
    'lat': 41.86,
    'lon': -87.67,
    'batch_count': cp3_config['batch_count'],
    'drift_threshold': cp3_config['drift_threshold'],
}

print(f'History rows: {len(history):,}')
print('Mock input:')
print(json.dumps(mock_input, indent=2))

## 3. Siapkan Data Input atau Mock Data

Notebook ini memakai data processed yang sudah tersedia. Untuk debugging cepat, kita juga menyiapkan mock input sederhana agar alur CP3 mudah diuji tanpa perlu menjalankan seluruh pipeline penuh.